# Chapter 9, Guide 2: HHL

Companion notebook to *Quantum Algorithms from First Principles*, Chapter 9,
"Building HHL" and "Testing HHL, and what the demonstrations really show".

Registers: one ancilla, $c$ clock qubits, $n$ system qubits. Step 0: choose $t_0$ so the
eigenvalues of $At_0/2\pi$ lie in $[0,1)$. Step 1: build $e^{iAt_0}$ (exact here, since
$A$ is one or two Paulis, or a precomputed unitary). Step 2: phase estimation with
controlled $e^{iAt_02^k}$, powers precomputed. Step 3: the rotation, hard-coded by
enumeration, $\theta=2\arcsin(C/\tilde\lambda)$ per clock value. Step 4: $\mathrm{QPE}^\dagger$,
measure the ancilla, post-select on $|1\rangle$.

Two cases: the widely reproduced $2\times2$ with eigenvalues $1$ and $2$, which the book
calls a teaching circuit rather than an implementation, and a $4\times4$ with eigenvalues
$1,2,4,8$ to exercise a 4-qubit clock.

Convention: book wire $j$ (top, most significant) is Qiskit qubit `bq(j)`; the clock's
wire $0$ is its most significant bit.

In [1]:
import numpy as np
from scipy.linalg import expm
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFTGate, UnitaryGate
from qiskit.quantum_info import Statevector, SparsePauliOp

bq = lambda j, tot: tot - 1 - j


def hhl_circuit(A, b, c, t0, C, ry_angle=None):
    """Ancilla = book wire 0, clock = wires 1..c, system = wires c+1..c+n."""
    n = int(np.log2(A.shape[0]))
    tot = 1 + c + n
    anc = bq(0, tot)
    clock = [bq(1 + k, tot) for k in range(c)]            # clock[0] most significant
    sysq = [bq(1 + c + j, tot) for j in range(n)]
    qc = QuantumCircuit(tot, name="HHL")

    # |b> on the system register (Statevector ordering: wire c+1 most significant -> reverse)
    qc.prepare_state(b / np.linalg.norm(b), sysq[::-1])

    # Step 2: QPE. clock wire k controls e^{iA t0 2^{c-1-k}}; powers precomputed exactly
    qc.h(clock)
    for k in range(c):
        power = 2 ** (c - 1 - k)
        U = expm(1j * A * t0 * power)
        qc.append(UnitaryGate(U, label=f"e^{{iAt0*{power}}}").control(1), [clock[k]] + sysq[::-1])
    qc.append(QFTGate(c).inverse(), clock[::-1])             # QFT^dagger, LSB-first list

    # Step 3: exact rotation by enumeration, one multi-controlled R_y per clock value
    for ell in range(1, 2**c):
        lam = 2 * np.pi * ell / (2**c * t0)                  # lambda tilde
        if C / lam > 1:                                      # clock values below C: skip (never populated)
            continue
        theta = 2 * np.arcsin(C / lam)
        bits = [(ell >> (c - 1 - k)) & 1 for k in range(c)]
        zeros = [clock[k] for k, bit in enumerate(bits) if bit == 0]
        if zeros: qc.x(zeros)
        qc.mcry(theta, clock, anc)
        if zeros: qc.x(zeros)

    # Step 4: uncompute QPE
    qc.append(QFTGate(c), clock[::-1])
    for k in reversed(range(c)):
        power = 2 ** (c - 1 - k)
        U = expm(-1j * A * t0 * power)
        qc.append(UnitaryGate(U, label=f"e^{{-iAt0*{power}}}").control(1), [clock[k]] + sysq[::-1])
    qc.h(clock)
    return qc, anc, clock, sysq


def analyze(A, b, c, t0, C):
    qc, anc, clock, sysq = hhl_circuit(A, b, c, t0, C)
    sv = Statevector(qc)
    n = len(sysq)
    # probabilities over (ancilla, clock) with the system traced out
    p_anc1 = sv.probabilities([anc])[1]
    p_clock = sv.probabilities(clock[::-1])                   # indexed by the clock integer
    # post-select ancilla = 1, clock = 0: project and read the system state
    amps = sv.data.reshape([2] * qc.num_qubits)              # axis 0 = highest Qiskit qubit = book wire 0 (ancilla)
    # axes in book order: ancilla, clock0..clock{c-1}, sys0..sys{n-1}
    sub = amps[1][(0,) * c]                                   # ancilla 1, clock all 0
    x_circ = sub.reshape(-1)                                  # system amplitudes, book order = integer index
    x_circ = x_circ / np.linalg.norm(x_circ)
    x_exact = np.linalg.solve(A, b); x_exact = x_exact / np.linalg.norm(x_exact)
    # compare up to global phase
    phase = np.vdot(x_exact, x_circ) / abs(np.vdot(x_exact, x_circ))
    fid = abs(np.vdot(x_exact, x_circ)) ** 2
    # success probability formula
    lam, U = np.linalg.eigh(A)
    bj = U.conj().T @ (b / np.linalg.norm(b))
    p_formula = sum(abs(bj) ** 2 * C**2 / lam**2)
    return dict(p_anc1=p_anc1, p_formula=p_formula, p_clock=p_clock, x_circ=x_circ * np.conj(phase), x_exact=x_exact, fid=fid, qc=qc)

## Case 1: the canonical $2\times2$, eigenvalues 1 and 2, clock of 2 qubits

$A=\begin{pmatrix}1.5&0.5\\0.5&1.5\end{pmatrix}$, $t_0=2\pi/4$ so that $\lambda t_0/2\pi=\lambda/4$ and the
clock holds $\tilde\lambda = 1, 2$ exactly; $C=\lambda_{\min}=1$.

In [2]:
A = np.array([[1.5, 0.5], [0.5, 1.5]])
b = np.array([1.0, 0.0])
c, t0, C = 2, 2 * np.pi / 4, 1.0
print("eigenvalues:", np.linalg.eigvalsh(A), " lambda*t0/2pi:", np.linalg.eigvalsh(A) * t0 / (2 * np.pi))

# Test 2: the clock holds the right eigenvalues before the rotation
qc_qpe, anc, clock, sysq = hhl_circuit(A, b, c, t0, C)
# rebuild just the QPE part to look at the clock
tot = 1 + c + 1
qc2 = QuantumCircuit(tot)
qc2.prepare_state(b, sysq[::-1]); qc2.h(clock)
for k in range(c):
    qc2.append(UnitaryGate(expm(1j * A * t0 * 2 ** (c - 1 - k))).control(1), [clock[k]] + sysq[::-1])
qc2.append(QFTGate(c).inverse(), clock[::-1])
pc = Statevector(qc2).probabilities(clock[::-1])
print("clock distribution after QPE:", np.round(pc, 4), " (lambda = 1 and 2 with weights |b_j|^2 = 0.5, 0.5)")
assert np.allclose(pc, [0, 0.5, 0.5, 0])

res = analyze(A, b, c, t0, C)
print(f"\nTest 3: clock back to |0> after QPE^dagger: {np.isclose(res['p_clock'][0], 1)}")
print(f"Test 4: fidelity with A^-1|b> normalized: {res['fid']:.10f}")
print("        circuit:", np.round(res["x_circ"], 5), " exact:", np.round(res["x_exact"], 5))
print(f"Test 5: Pr(ancilla=1) = {res['p_anc1']:.6f}, formula sum |b_j|^2 C^2/lambda_j^2 = {res['p_formula']:.6f}")
assert res["fid"] > 1 - 1e-10 and abs(res["p_anc1"] - res["p_formula"]) < 1e-10
print("all tests passed")

eigenvalues: [1. 2.]  lambda*t0/2pi: [0.25 0.5 ]
clock distribution after QPE: [0.  0.5 0.5 0. ]  (lambda = 1 and 2 with weights |b_j|^2 = 0.5, 0.5)

Test 3: clock back to |0> after QPE^dagger: True
Test 4: fidelity with A^-1|b> normalized: 1.0000000000
        circuit: [ 0.94868+0.j -0.31623+0.j]  exact: [ 0.94868 -0.31623]
Test 5: Pr(ancilla=1) = 0.625000, formula sum |b_j|^2 C^2/lambda_j^2 = 0.625000
all tests passed


## Case 2: $4\times4$ with eigenvalues $1,2,4,8$ and a 4-qubit clock

Eigenvalues that are exact powers of two make phase estimation exact, so any error is
yours (test 1). $t_0 = 2\pi/16$, $C=1$, so the success probability is small: that is the
$1/\kappa^2$ the book warns about, with $\kappa=8$.

In [3]:
rng = np.random.default_rng(3)
Uq, _ = np.linalg.qr(rng.normal(size=(4, 4)) + 1j * rng.normal(size=(4, 4)))
A4 = Uq @ np.diag([1.0, 2.0, 4.0, 8.0]) @ Uq.conj().T
b4 = rng.normal(size=4) + 1j * rng.normal(size=4)
c4, t04, C4 = 4, 2 * np.pi / 16, 1.0
res4 = analyze(A4, b4, c4, t04, C4)
print(f"clock back to |0>: {np.isclose(res4['p_clock'][0], 1)}")
print(f"fidelity with A^-1|b>: {res4['fid']:.10f}")
print(f"Pr(ancilla=1) = {res4['p_anc1']:.5f}, formula {res4['p_formula']:.5f}, bound 1/kappa^2 = {1/64:.5f}")
assert res4["fid"] > 1 - 1e-9
print("qubits:", res4["qc"].num_qubits, "(1 ancilla + 4 clock + 2 system); multi-controlled R_y gates:", 2**c4 - 1)

clock back to |0>: True
fidelity with A^-1|b>: 1.0000000000
Pr(ancilla=1) = 0.39311, formula 0.39311, bound 1/kappa^2 = 0.01562
qubits: 7 (1 ancilla + 4 clock + 2 system); multi-controlled R_y gates: 15


## What breaks when $t_0$ is wrong

Take $t_0$ so that $\lambda t_0/2\pi$ wraps past 1: phase estimation returns the fractional
part and the rotation inverts the wrong number.

In [4]:
res_bad = analyze(A, b, c, 2 * np.pi / 2, C)       # lambda=2 -> phase 1.0 -> wraps to 0
print(f"t0 = 2pi/2: fidelity {res_bad['fid']:.4f}, clock-zero probability after uncompute {res_bad['p_clock'][0]:.4f}")

t0 = 2pi/2: fidelity 0.8000, clock-zero probability after uncompute 1.0000


The $2\times2$ case is exact because its eigenvalues are chosen to be representable on a
2-qubit clock and the angles are hard-coded. It demonstrates the structure, not a
scalable linear solver: a general $A$ needs Hamiltonian simulation for $e^{iAt_0}$
(Chapter 7), a clock wide enough for $\log_2\kappa+\log_2(1/\epsilon)$ bits, and
arithmetic for $1/\tilde\lambda$ in place of the $2^c-1$ enumerated rotations.